In [1]:
from unet.hexunet_parts import *
from unet.hexunet_model import HexUNet

In [2]:
import torch


# ---------------------------------------------------------
# Dummy input
# ---------------------------------------------------------

x = torch.arange(
    1,
    26,
    dtype=torch.float32,
).reshape(1, 1, 5, 5)

print("Input:")
print(x)
print("Shape:", x.shape)


# ---------------------------------------------------------
# Test hex kernel masks
# ---------------------------------------------------------

for kernel_size in [3, 5, 7]:

    mask = hex_kernel_mask(kernel_size)

    print(f"\nKernel mask {kernel_size}x{kernel_size}:")
    print(mask)

    radius = kernel_size // 2
    expected_cells = 1 + 3 * radius * (radius + 1)

    print("Active cells:", int(mask.sum()))
    print("Expected:", expected_cells)

    assert int(mask.sum()) == expected_cells


# ---------------------------------------------------------
# Test HConv2d
# ---------------------------------------------------------

conv = HConv2d(
    in_channels=1,
    out_channels=4,
    kernel_size=3,
    padding=1,
)

y = conv(x)

print("\nHConv2d:")
print("Input shape:", x.shape)
print("Output shape:", y.shape)

assert y.shape == (1, 4, 5, 5)


# ---------------------------------------------------------
# Test HDoubleConv
# ---------------------------------------------------------

double_conv = HDoubleConv(
    in_channels=1,
    out_channels=8,
)

y = double_conv(x)

print("\nHDoubleConv:")
print("Input shape:", x.shape)
print("Output shape:", y.shape)

assert y.shape == (1, 8, 5, 5)


# ---------------------------------------------------------
# Test hmax_pool2d
# ---------------------------------------------------------

pooled = hmax_pool2d(
    x,
    stride=2,
)

print("\nhmax_pool2d:")
print(pooled)
print("Shape:", pooled.shape)


# ---------------------------------------------------------
# Test HMaxPool2d
# ---------------------------------------------------------

pool = HMaxPool2d(
    stride=2,
)

pooled_module = pool(x)

print("\nHMaxPool2d:")
print(pooled_module)
print("Shape:", pooled_module.shape)

assert torch.equal(
    pooled,
    pooled_module,
)


# ---------------------------------------------------------
# Test h_upsample2d
# ---------------------------------------------------------

upsampled = h_upsample2d(
    pooled
)

print("\nh_upsample2d:")
print(upsampled)
print("Shape:", upsampled.shape)


# ---------------------------------------------------------
# Test HUpsample
# ---------------------------------------------------------

upsample = HUpsample()

upsampled_module = upsample(
    pooled
)

print("\nHUpsample:")
print(upsampled_module)
print("Shape:", upsampled_module.shape)

assert torch.equal(
    upsampled,
    upsampled_module,
)


# ---------------------------------------------------------
# Test HDown
# ---------------------------------------------------------

down = HDown(
    in_channels=1,
    out_channels=8,
)

down_output = down(x)

print("\nHDown:")
print("Input shape:", x.shape)
print("Output shape:", down_output.shape)


# ---------------------------------------------------------
# Test HUp
#
# Use HDown output as the decoder input.
# Use an encoder tensor with the target spatial resolution
# as the skip connection.
# ---------------------------------------------------------

skip = torch.randn(
    1,
    8,
    5,
    5,
)

x1 = torch.randn(
    1,
    16,
    3,
    3,
)

up = HUp(
    in_channels=16,
    out_channels=8,
)

up_output = up(
    x1,
    skip,
)

print("\nHUp:")
print("Decoder input:", x1.shape)
print("Skip input:", skip.shape)
print("Output:", up_output.shape)


print("\nAll tests completed.")

Input:
tensor([[[[ 1.,  2.,  3.,  4.,  5.],
          [ 6.,  7.,  8.,  9., 10.],
          [11., 12., 13., 14., 15.],
          [16., 17., 18., 19., 20.],
          [21., 22., 23., 24., 25.]]]])
Shape: torch.Size([1, 1, 5, 5])

Kernel mask 3x3:
tensor([[0., 1., 1.],
        [1., 1., 1.],
        [1., 1., 0.]])
Active cells: 7
Expected: 7

Kernel mask 5x5:
tensor([[0., 0., 1., 1., 1.],
        [0., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 0., 0.]])
Active cells: 19
Expected: 19

Kernel mask 7x7:
tensor([[0., 0., 0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.]])
Active cells: 37
Expected: 37

HConv2d:
Input shape: torch.Size([1, 1, 5, 5])
Output shape: torch.Size([1, 4, 5, 5])

HDoubleConv:
Input shape: torch.Size([1, 1, 5, 5])
Output 

In [3]:
import torch
import torch.nn.functional as F

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

batch_size = 2
n_channels = 1
n_classes = 1
height = 64
width = 64


# ---------------------------------------------------------
# Create model
# ---------------------------------------------------------

model = HexUNet(
    n_channels=n_channels,
    n_classes=n_classes,
).to(device)

print("Device:", device)


# ---------------------------------------------------------
# Create dummy input
#
# Shape:
#   [batch, channels, height, width]
# ---------------------------------------------------------

x = torch.randn(
    batch_size,
    n_channels,
    height,
    width,
    device=device,
)

print("Input shape:", x.shape)


# ---------------------------------------------------------
# Forward pass
# ---------------------------------------------------------

output = model(x)

print("Output shape:", output.shape)


# ---------------------------------------------------------
# Validate output shape
# ---------------------------------------------------------

expected_shape = (
    batch_size,
    n_classes,
    height,
    width,
)

assert output.shape == expected_shape, (
    f"Expected output shape {expected_shape}, "
    f"got {tuple(output.shape)}"
)


# ---------------------------------------------------------
# Check for invalid values
# ---------------------------------------------------------

assert torch.isfinite(output).all(), (
    "Output contains NaN or infinite values"
)

print("Output values are finite.")


# ---------------------------------------------------------
# Create dummy target
# ---------------------------------------------------------

target = torch.randn_like(output)


# ---------------------------------------------------------
# Calculate loss
# ---------------------------------------------------------

loss = F.mse_loss(
    output,
    target,
)

print("Loss:", loss.item())

assert torch.isfinite(loss), (
    "Loss is NaN or infinite"
)


# ---------------------------------------------------------
# Backward pass
# ---------------------------------------------------------

model.zero_grad()

loss.backward()

print("Backward pass completed.")


# ---------------------------------------------------------
# Verify that gradients reached model parameters
# ---------------------------------------------------------

parameters_with_grad = 0
parameters_without_grad = 0

for parameter in model.parameters():

    if not parameter.requires_grad:
        continue

    if parameter.grad is None:
        parameters_without_grad += 1
    else:
        parameters_with_grad += 1


print(
    "Parameters with gradients:",
    parameters_with_grad,
)

print(
    "Parameters without gradients:",
    parameters_without_grad,
)

assert parameters_with_grad > 0, (
    "No model parameters received gradients"
)


# ---------------------------------------------------------
# Check gradients for invalid values
# ---------------------------------------------------------

for name, parameter in model.named_parameters():

    if parameter.grad is None:
        continue

    assert torch.isfinite(parameter.grad).all(), (
        f"Invalid gradient detected in {name}"
    )

print("All gradients are finite.")


# ---------------------------------------------------------
# Perform one optimizer step
# ---------------------------------------------------------

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

optimizer.step()

print("Optimizer step completed.")


# ---------------------------------------------------------
# Run another forward pass after parameter update
# ---------------------------------------------------------

with torch.no_grad():

    output_after_update = model(x)

assert torch.isfinite(
    output_after_update
).all()

print(
    "Output shape after update:",
    output_after_update.shape,
)


print("\nHexUNet test passed.")

Device: cpu
Input shape: torch.Size([2, 1, 64, 64])
x1: torch.Size([2, 64, 64, 64])
x2: torch.Size([2, 128, 32, 32])
x3: torch.Size([2, 256, 16, 16])
x4: torch.Size([2, 512, 8, 8])
x5: torch.Size([2, 1024, 4, 4])
up1: torch.Size([2, 512, 8, 8])
up2: torch.Size([2, 256, 16, 16])
up3: torch.Size([2, 128, 32, 32])
up4: torch.Size([2, 64, 64, 64])
Output shape: torch.Size([2, 1, 64, 64])
Output values are finite.
Loss: 1.1838724613189697
Backward pass completed.
Parameters with gradients: 64
Parameters without gradients: 0
All gradients are finite.
Optimizer step completed.
x1: torch.Size([2, 64, 64, 64])
x2: torch.Size([2, 128, 32, 32])
x3: torch.Size([2, 256, 16, 16])
x4: torch.Size([2, 512, 8, 8])
x5: torch.Size([2, 1024, 4, 4])
up1: torch.Size([2, 512, 8, 8])
up2: torch.Size([2, 256, 16, 16])
up3: torch.Size([2, 128, 32, 32])
up4: torch.Size([2, 64, 64, 64])
Output shape after update: torch.Size([2, 1, 64, 64])

HexUNet test passed.
